In [1]:
import random
import torch


# ============================================================
# 0. 固定随机种子
# 这样每次运行时生成的数据基本一致，方便观察和排错
# ============================================================
random.seed(42)
torch.manual_seed(42)


# ============================================================
# 1. 生成人工数据
#
# 我们假装真实规律是：
# y = 2 * x1 - 3.4 * x2 + 4.2 + 噪声
#
# 模型不知道 2、-3.4、4.2，
# 它需要通过数据把这些参数学习出来。
# ============================================================
def synthetic_data(w, b, num_examples):
    """
    w: 真实权重，例如 tensor([2.0, -3.4])
    b: 真实偏置，例如 4.2
    num_examples: 生成多少条数据
    """

    # 生成特征 X
    # 形状为：(样本数量, 特征数量)
    # 此处是：(1000, 2)
    X = torch.normal(
        mean=0,
        std=1,
        size=(num_examples, len(w))
    )

    # 根据真实公式生成标签
    # X 的形状：(1000, 2)
    # w 的形状：(2,)
    # X @ w 的形状：(1000,)
    y = X @ w + b

    # 加入少量随机噪声，让数据更接近真实世界
    y += torch.normal(
        mean=0,
        std=0.01,
        size=y.shape
    )

    # 把 y 从 (1000,) 改为 (1000, 1)
    return X, y.reshape(-1, 1)


# 设置真实参数
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2

# 生成 1000 条人工数据
features, labels = synthetic_data(
    true_w,
    true_b,
    num_examples=1000
)

print("features 的形状：", features.shape)
print("labels 的形状：", labels.shape)

print("\n第一条特征：", features[0])
print("第一条标签：", labels[0])


# ============================================================
# 2. 自己实现小批量数据迭代器
#
# 每次不把 1000 条数据全部交给模型，
# 而是每次取 batch_size 条。
# ============================================================
def data_iter(batch_size, features, labels):
    """
    batch_size: 每个批次包含多少条数据
    features: 所有特征
    labels: 所有标签
    """

    num_examples = len(features)

    # 得到所有样本编号：
    # [0, 1, 2, ..., 999]
    indices = list(range(num_examples))

    # 打乱样本顺序
    random.shuffle(indices)

    # 每次取 batch_size 个样本
    for i in range(0, num_examples, batch_size):
        batch_indices = indices[i:i + batch_size]

        # 把 Python 列表转换成 PyTorch 张量
        batch_indices = torch.tensor(batch_indices)

        # yield 每次返回一个小批量
        yield features[batch_indices], labels[batch_indices]


batch_size = 10

# 先取出一个批次，观察形状
for X, y in data_iter(batch_size, features, labels):
    print("\n一个批次中 X 的形状：", X.shape)
    print("一个批次中 y 的形状：", y.shape)
    break


# ============================================================
# 3. 初始化模型要学习的参数
#
# 注意：
# true_w 和 true_b 是生成数据时使用的真实答案；
# w 和 b 是模型一开始随机猜测的答案。
# ============================================================

# w 的形状是 (2, 1)
# requires_grad=True 表示需要计算它的梯度
w = torch.normal(
    mean=0,
    std=0.01,
    size=(2, 1),
    requires_grad=True
)

# b 的形状是 (1,)
b = torch.zeros(
    1,
    requires_grad=True
)

print("\n初始 w：")
print(w)

print("初始 b：")
print(b)


# ============================================================
# 4. 定义线性回归模型
#
# 数学公式：
# y_hat = Xw + b
# ============================================================
def linreg(X, w, b):
    """
    X: 一个批次的特征
    w: 模型权重
    b: 模型偏置
    """

    return X @ w + b


# ============================================================
# 5. 定义平方损失函数
#
# loss = 1/2 * (预测值 - 真实值)^2
#
# 除以 2 主要是为了求导后形式更简洁，
# 不除以 2 也可以训练。
# ============================================================
def squared_loss(y_hat, y):
    """
    y_hat: 模型预测值
    y: 真实标签
    """

    # 保证 y 的形状和 y_hat 一致
    y = y.reshape(y_hat.shape)

    return (y_hat - y) ** 2 / 2


# ============================================================
# 6. 手动实现小批量随机梯度下降 SGD
#
# 参数更新公式：
# 参数 = 参数 - 学习率 * 梯度
# ============================================================
def sgd(params, lr, batch_size):
    """
    params: 需要更新的参数，例如 [w, b]
    lr: 学习率
    batch_size: 批量大小
    """

    # 更新参数的操作本身不需要计算梯度
    with torch.no_grad():

        for param in params:
            # 前面使用的是 loss.sum().backward()
            # 因此这里除以 batch_size，得到一个批次的平均梯度
            param -= lr * param.grad / batch_size

            # PyTorch 默认会累加梯度
            # 每次更新后必须把梯度清零
            param.grad.zero_()


# ============================================================
# 7. 设置训练超参数
# ============================================================
learning_rate = 0.03
num_epochs = 3
batch_size = 10


# ============================================================
# 8. 开始训练
# ============================================================
for epoch in range(num_epochs):

    # 每个 epoch 都会遍历一次全部训练数据
    for X, y in data_iter(batch_size, features, labels):

        # 第一步：模型进行预测
        y_hat = linreg(X, w, b)

        # 第二步：计算每个样本的损失
        loss = squared_loss(y_hat, y)

        # 第三步：反向传播，计算 w 和 b 的梯度
        #
        # loss 的形状是 (batch_size, 1)
        # backward 通常需要一个标量
        # 所以先使用 sum() 把它们加起来
        loss.sum().backward()

        # 第四步：根据梯度更新参数，并清空梯度
        sgd(
            params=[w, b],
            lr=learning_rate,
            batch_size=X.shape[0]
        )

    # 一个 epoch 结束后，计算全部数据的平均损失
    with torch.no_grad():
        train_loss = squared_loss(
            linreg(features, w, b),
            labels
        )

        print(
            f"epoch {epoch + 1}, "
            f"loss {train_loss.mean().item():.6f}"
        )


# ============================================================
# 9. 比较模型学到的参数和真实参数
# ============================================================
print("\n真实 w：")
print(true_w)

print("模型学习到的 w：")
print(w.reshape(-1))

print("\n真实 b：")
print(true_b)

print("模型学习到的 b：")
print(b.item())

print("\nw 的误差：")
print(true_w - w.reshape(-1))

print("b 的误差：")
print(true_b - b.item())

features 的形状： torch.Size([1000, 2])
labels 的形状： torch.Size([1000, 1])

第一条特征： tensor([1.9269, 1.4873])
第一条标签： tensor([2.9871])

一个批次中 X 的形状： torch.Size([10, 2])
一个批次中 y 的形状： torch.Size([10, 1])

初始 w：
tensor([[ 0.0114],
        [-0.0007]], requires_grad=True)
初始 b：
tensor([0.], requires_grad=True)
epoch 1, loss 0.047187
epoch 2, loss 0.000199
epoch 3, loss 0.000050

真实 w：
tensor([ 2.0000, -3.4000])
模型学习到的 w：
tensor([ 1.9994, -3.3998], grad_fn=<ViewBackward0>)

真实 b：
4.2
模型学习到的 b：
4.199854850769043

w 的误差：
tensor([ 0.0006, -0.0002], grad_fn=<SubBackward0>)
b 的误差：
0.00014514923095720889


In [2]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader


# 1. 生成人工数据
def synthetic_data(w, b, num_examples):
    X = torch.normal(0, 1, size=(num_examples, len(w)))
    y = X @ w + b
    y += torch.normal(0, 0.01, size=y.shape)
    return X, y.reshape(-1, 1)


true_w = torch.tensor([2.0, -3.4])
true_b = 4.2

features, labels = synthetic_data(true_w, true_b, 1000)


In [3]:
dataset = TensorDataset(features,labels)
data_loader = DataLoader(
    dataset,
    batch_size=10,
    shuffle=True
)
model = nn.Linear(2,1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr = 0.03
)

In [6]:
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for X,y in data_loader:
        optimizer.zero_grad()
        y_hat = model(X)
        loss = loss_fn(y_hat,y)
        loss.backward()
        optimizer.step()

    print(
        f"epoch {epoch + 1}, "
        f"loss = {loss.item():.6f}"
    )  

epoch 1, loss = 0.000165
epoch 2, loss = 0.000136
epoch 3, loss = 0.000198


In [7]:
# 7. 查看学习结果
print("真实 w：", true_w)
print("学习 w：", model.weight.data.reshape(-1))

print("真实 b：", true_b)
print("学习 b：", model.bias.data.item())

真实 w： tensor([ 2.0000, -3.4000])
学习 w： tensor([ 1.9999, -3.4008])
真实 b： 4.2
学习 b： 4.199979782104492
